In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import re

# === Scientific Publication Style ===
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'axes.linewidth': 1.2,
    'axes.grid': True,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'legend.fontsize': 10,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '#cccccc',
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

In [ ]:
# ========= 1. 實驗設定 (Configuration) =========

base_dir = r"E:\experiment\2D"

experiments = [
    {"file": os.path.join(base_dir, "t0185", "t0185.csv"), "group_val": 10, "group": "10cm"},
    {"file": os.path.join(base_dir, "t0184", "t0184.csv"), "group_val": 12.5, "group": "12.5cm"},
    {"file": os.path.join(base_dir, "t0183", "t0183.csv"), "group_val": 15, "group": "15cm"},
    # {"file": os.path.join(base_dir, "t0141", "t0141.csv"), "group_val": 20, "group": "20cm"}
]

# 圖表 X 軸 (Normal Stress) 顯示範圍
xlim_min = 1
xlim_max = 9

In [ ]:
# ========= 2. 資料處理 (Data Processing) =========

def extract_sigma(run_name):
    '''從 run1_32MPa 這樣的字串中提取 32 作為數值'''
    match = re.search(r'_(\d+)MPa', str(run_name))
    if match:
        return float(match.group(1))
    return np.nan


def process_data(target_metric):
    '''讀取 CSV 並計算各組在每個 sigma 下的 median 與 IQR'''
    processed_data = []
    for exp in experiments:
        if not os.path.exists(exp['file']):
            print(f"找不到檔案: {exp['file']}")
            continue

        df = pd.read_csv(exp['file'])

        if 'skipped' in df.columns:
            df = df[df['skipped'] != 'YES'].copy()

        if target_metric == 'k' and 'k' in df.columns:
            df = df[pd.to_numeric(df['k'], errors='coerce') >= 0]

        if target_metric == 'shear_force' and 'delta_tau' in df.columns:
            # Force [kN] = delta_tau [MPa] * pi * r[cm]^2 * 0.1
            df['shear_force'] = df['delta_tau'] * np.pi * (exp['group_val'] ** 2) * 0.1

        if target_metric == 'delta_mu' and 'delta_mu' in df.columns:
            # mu_local = mu_global * (A_global / A_local)
            # A_global = 50 * 50 = 2500 cm^2
            # A_local = pi * r^2
            df['delta_mu'] = df['delta_mu'] * (2500 / (np.pi * (exp['group_val'] ** 2)))

        if target_metric not in df.columns:
            continue

        # 提取 sigma
        df['sigma'] = df['Run'].apply(extract_sigma)
        df = df.dropna(subset=['sigma', target_metric])

        # 計算各個 sigma 的統計量
        stats = df.groupby('sigma')[target_metric].agg(
            median='median',
            q1=lambda x: x.quantile(0.25),
            q3=lambda x: x.quantile(0.75)
        ).reset_index()

        processed_data.append({
            'group': exp['group'],
            'group_val': exp['group_val'],
            'stats': stats
        })
    return processed_data

In [ ]:
# Paul Tol's colorblind-friendly palette
custom_colors = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#AA4499']
markers = ['o', 's', '^', 'D', 'v']

In [ ]:
# ========= 3. 繪製圖表函數 (Plotting Function) =========

def plot_metric(target_metric, title, ylabel, loglog=False):
    data_to_plot = process_data(target_metric)
    if not data_to_plot:
        print(f'找不到 {target_metric} 的資料')
        return

    n = len(data_to_plot)
    colors = custom_colors[:n]

    fig, ax = plt.subplots(figsize=(7, 5))

    all_sigmas = []
    for i, item in enumerate(data_to_plot):
        stats = item['stats']
        sigmas = stats['sigma'].values
        medians = stats['median'].values
        yerr_lower = medians - stats['q1'].values
        yerr_upper = stats['q3'].values - medians
        color = colors[i]
        marker = markers[i % len(markers)]

        all_sigmas.extend(sigmas)

        ax.errorbar(
            sigmas, medians,
            yerr=[yerr_lower, yerr_upper],
            fmt=marker, color=color,
            markersize=8, capsize=4,
            markeredgecolor='white', markeredgewidth=0.8,
            capthick=1.2, elinewidth=1.2, zorder=3,
            label=item['group']
        )

        # Trend line (不標示方程式)
        if target_metric in ['delta_tau', 'shear_force', 'delta_mu'] and len(sigmas) > 1:
            from scipy.optimize import curve_fit
            def power_law(x, a, b): return a * (x ** b)
            slope_init, intercept_init = np.polyfit(np.log(sigmas), np.log(medians), 1)
            p0 = [np.exp(intercept_init), slope_init]
            try:
                popt, _ = curve_fit(power_law, sigmas, medians, p0=p0, maxfev=10000)
                a, b = popt
            except:
                a, b = p0[0], p0[1]
            x_line = np.linspace(sigmas.min(), sigmas.max(), 50)
            y_line = a * (x_line ** b)
            ax.plot(x_line, y_line, color=color, linestyle='-',
                    linewidth=1.5, alpha=0.6, zorder=2)
        elif len(sigmas) > 1:
            slope, intercept = np.polyfit(sigmas, medians, 1)
            x_line = np.array([sigmas.min(), sigmas.max()])
            y_line = slope * x_line + intercept
            ax.plot(x_line, y_line, color=color, linestyle='-',
                    linewidth=1.5, alpha=0.6, zorder=2)

    if loglog:
        ax.set_xscale('log')
        ax.set_yscale('log')
        from matplotlib.ticker import ScalarFormatter, NullFormatter
        ax.xaxis.set_major_formatter(ScalarFormatter())
        ax.yaxis.set_major_formatter(ScalarFormatter())
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.yaxis.set_minor_formatter(NullFormatter())

    ax.set_xlabel(r'Normal Stress $\sigma$ [MPa]')
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    if loglog and all_sigmas:
        ax.set_xlim(min(all_sigmas) * 0.8, max(all_sigmas) * 1.2)
    else:
        try:
            ax.set_xlim(xlim_min, xlim_max)
        except:
            pass

    if all_sigmas:
        ax.set_xticks(np.unique(all_sigmas))

    ax.legend(title='Seismogenic Zone Radius', loc='best')
    fig.tight_layout()
    plt.show()

In [ ]:
# ========= 4. 以群組為 X 軸的繪圖函數 =========

def plot_metric_by_group(target_metric, title, ylabel, loglog=False):
    '''以實驗群組為 X 軸（順序與 experiments 設定相同），不同 sigma 為系列繪圖。'''
    group_order = [exp['group'] for exp in experiments]
    group_xval = {exp['group']: exp['group_val'] for exp in experiments}

    data_to_plot = process_data(target_metric)
    if not data_to_plot:
        print(f'找不到 {target_metric} 欄位')
        return

    # 收集所有出現的 sigma 值
    all_sigmas_set = set()
    for item in data_to_plot:
        all_sigmas_set.update(item['stats']['sigma'].values)
    unique_sigmas = sorted(all_sigmas_set)

    n = len(unique_sigmas)
    colors = custom_colors[:n]

    fig, ax = plt.subplots(figsize=(7, 5))

    for j, sigma in enumerate(unique_sigmas):
        x_vals, y_vals, yerr_lo, yerr_hi = [], [], [], []

        for item in data_to_plot:
            group = item['group']
            if group not in group_xval:
                continue
            stats = item['stats']
            row = stats[stats['sigma'] == sigma]
            if row.empty:
                continue
            x_vals.append(group_xval[group])
            med = row['median'].values[0]
            q1 = row['q1'].values[0]
            q3 = row['q3'].values[0]
            y_vals.append(med)
            yerr_lo.append(med - q1)
            yerr_hi.append(q3 - med)

        if not x_vals:
            continue

        x_arr = np.array(x_vals)
        y_arr = np.array(y_vals)
        color = colors[j]
        marker = markers[j % len(markers)]

        ax.errorbar(
            x_arr, y_arr,
            yerr=[np.array(yerr_lo), np.array(yerr_hi)],
            fmt=marker, color=color,
            markersize=8, capsize=4,
            markeredgecolor='white', markeredgewidth=0.8,
            capthick=1.2, elinewidth=1.2, zorder=3,
            label=f'{sigma:.0f} MPa'
        )

        # Trend line (不標示方程式)
        if target_metric in ['delta_tau', 'shear_force', 'delta_mu'] and len(x_arr) > 1:
            from scipy.optimize import curve_fit
            def power_law(x, a, b): return a * (x ** b)
            slope_init, intercept_init = np.polyfit(np.log(x_arr), np.log(y_arr), 1)
            p0 = [np.exp(intercept_init), slope_init]
            try:
                popt, _ = curve_fit(power_law, x_arr, y_arr, p0=p0, maxfev=10000)
                a, b = popt
            except:
                a, b = p0[0], p0[1]
            x_line = np.linspace(x_arr.min(), x_arr.max(), 50)
            y_line = a * (x_line ** b)
            ax.plot(x_line, y_line, color=color, linestyle='-',
                    linewidth=1.5, alpha=0.6, zorder=2)
        elif len(x_arr) > 1:
            slope, intercept = np.polyfit(x_arr, y_arr, 1)
            x_line = np.array([x_arr.min(), x_arr.max()])
            y_line = slope * x_line + intercept
            ax.plot(x_line, y_line, color=color, linestyle='-',
                    linewidth=1.5, alpha=0.6, zorder=2)

    if loglog:
        ax.set_xscale('log')
        ax.set_yscale('log')
        from matplotlib.ticker import ScalarFormatter, NullFormatter
        ax.xaxis.set_major_formatter(ScalarFormatter())
        ax.yaxis.set_major_formatter(ScalarFormatter())
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.yaxis.set_minor_formatter(NullFormatter())

    ax.set_xlabel('Seismogenic Zone Radius')
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    x_vals_all = list(group_xval.values())
    if loglog:
        ax.set_xlim(min(x_vals_all) * 0.8, max(x_vals_all) * 1.2)
    else:
        ax.set_xlim(min(x_vals_all) - 1, max(x_vals_all) + 1)
    ax.set_xticks(x_vals_all)
    ax.set_xticklabels(group_order)

    ax.legend(title=r'Normal Stress $\sigma$', loc='best')
    fig.tight_layout()
    plt.show()

## X = Normal Stress σ, Series = Seismogenic Zone Radius

In [ ]:
plot_metric('delta_tau', r'$\Delta\tau$ vs Normal Stress', r'$\Delta\tau$ [MPa]', loglog=True)

In [ ]:
plot_metric('k', r'Stiffness $k$ vs Normal Stress', r'$k$')

In [ ]:
plot_metric('shear_force', r'Shear Force vs Normal Stress', r'$\Delta F$ [kN]', loglog=True)

In [ ]:
plot_metric('delta_mu', r'$\Delta\mu_{local}$ vs Normal Stress', r'$\Delta\mu_{local}$', loglog=True)

## X = Seismogenic Zone Radius, Series = Normal Stress σ

In [ ]:
plot_metric_by_group('delta_tau', r'$\Delta\tau$ vs Seismogenic Zone Radius', r'$\Delta\tau$ [MPa]', loglog=True)

In [ ]:
plot_metric_by_group('k', r'Stiffness $k$ vs Seismogenic Zone Radius', r'$k$')

In [ ]:
plot_metric_by_group('shear_force', r'Shear Force vs Seismogenic Zone Radius', r'$\Delta F$ [kN]', loglog=True)

In [ ]:
plot_metric_by_group('delta_mu', r'$\Delta\mu_{local}$ vs Seismogenic Zone Radius', r'$\Delta\mu_{local}$', loglog=True)